# 5. Larger networks on a GPU (Colab)

The committed results are CPU-scale by necessity: ~54-node networks, 10 epochs,
a 3-member ensemble, all on two threads of a shared 4-core machine. This
notebook runs the **same code** at a scale the CPU budget cannot reach.

Nothing here is required to reproduce a documented number. It exists so a
reader with a GPU can check whether the conclusions survive more capacity —
and the honest expectation is that the *rank* results improve more than the
magnitude ones.

## Setup

On Colab, uncomment the clone cell. Locally, this runs from the repo root.

In [ ]:
# !git clone https://github.com/HabibaSajid321/supply-network-disruption-surrogate.git
# %cd supply-network-disruption-surrogate
# !pip install -q torch numpy scipy pandas pyyaml matplotlib scikit-learn
import os, sys
sys.path.insert(0, '../src' if os.path.basename(os.getcwd())=='notebooks' else 'src')
import torch
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', DEVICE, '| torch', torch.__version__)
if DEVICE == 'cpu':
    print('No GPU: the cells below will still run but use the small settings.')

## Scale up

Larger networks, more scenarios, a wider model, more epochs, a bigger ensemble.
Note that **dataset generation is CPU-bound** — the simulator is pure Python —
so a GPU speeds up training but not data generation. That asymmetry is exactly
why the break-even accounting in notebook 4 charges dataset time separately.

In [ ]:
from sndsur.config import load_config
big = ['netgen.n_per_tier=[40,34,24,14,18]',
       'dataset.n_train_networks=20',
       'dataset.scenarios_per_train_network=500',
       'dataset.max_train_scenarios=0',
       'model.hidden=128', 'model.layers=4', 'model.traj_horizon=24',
       'model.ensemble=5', 'train.epochs=60', 'train.batch_graphs=32',
       f'run.device={DEVICE}', 'run.threads=8', 'run.name=gpu_full']
cfg = load_config('../configs/base.yaml' if os.path.basename(os.getcwd())=='notebooks' else 'configs/base.yaml', big)
print(cfg.netgen.n_per_tier, '->', sum(cfg.netgen.n_per_tier), 'nodes per network')
print('hidden', cfg.model.hidden, '| layers', cfg.model.layers, '| ensemble', cfg.model.ensemble)

In [ ]:
# Dataset generation is the slow part (CPU-bound simulator). Expect ~20-40 min.
# from sndsur.pipelines import build_data
# build_data(cfg, rebuild=True)

In [ ]:
# from sndsur.pipelines import run_comparison, run_criticality, run_efficiency
# frames = run_comparison(cfg)
# frames['methods'][['split','method','mae','spearman_within_scenario']].round(4)

## Bring your own network

`sndsur.data.csvio` loads a user-supplied topology from CSV. The **mechanics**
are still this simulator's mechanics, so results transfer only insofar as those
mechanics match reality. No real-world validation is claimed anywhere in this
repository.

In [ ]:
from sndsur.data.csvio import write_example_csvs, load_network_from_csv
import tempfile, pathlib
d = pathlib.Path(tempfile.mkdtemp())
write_example_csvs(d)
print(open(d/'nodes.csv').read())
print(open(d/'edges.csv').read())
net = load_network_from_csv(d/'nodes.csv', d/'edges.csv')
net.validate()
print('loaded:', net.n_nodes, 'nodes,', len(net.edges()), 'edges')